# NB3 — Réduction, sélection et alternative probabiliste

Notebook des pipelines **P11 à P15**.

## Portée du notebook

Ce notebook analyse l'impact de la **maîtrise de la dimension** :
- compression par `TruncatedSVD` ;
- sélection supervisée par `chi²` ;
- variante probabiliste avec `ComplementNB`.

Les évaluations sont faites directement sur le couple `train.csv` / `test.csv` déjà préparé.

In [1]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow

from collections import OrderedDict

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import Normalizer

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    evaluate_sklearn_pipeline,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
    GensimMeanEmbeddingVectorizer,
    SentenceTransformerVectorizer,
)

seed_everything(42)
import mlflow


from mlflow_utils import (
    setup_mlflow_tracking,
    fit_evaluate_and_log_sklearn_pipeline,
)


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [2]:
DATA_DIR = "../../data/processed_data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB3_reduction_selection_nb"
RESULTS_DIR = "../../outputs/NB3"

# Configuration MLflow
from pathlib import Path
MLFLOW_EXPERIMENT_NAME = "DT_NB3_reduction_selection_nb"
MLFLOW_TRACKING_URI = Path("../../outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/06 22:22:03 INFO mlflow.tracking.fluent: Experiment with name 'DT_NB3_reduction_selection_nb' does not exist. Creating a new experiment.


MLflow tracking URI : file:///C:/Users/DELL/Documents/Classes/ISE2/ISE2_2026/SEM2/ML2/Projet/Disaster-Tweets-NLP/outputs/mlruns
MLflow experiment   : DT_NB3_reduction_selection_nb


In [3]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())

Taille train : 9096
Taille test  : 2274

Distribution des classes - train :
target
0    0.814094
1    0.185906
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


In [4]:
pipelines = OrderedDict({
    "P11_TFIDF_SVD_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("norm", Normalizer(copy=False)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P12_TFIDF_SVD_LinearSVC": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("norm", Normalizer(copy=False)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P13_TFIDF_SelectKBest_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("select", SelectKBest(score_func=chi2, k=5000)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P14_TFIDF_SelectKBest_LinearSVC": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("select", SelectKBest(score_func=chi2, k=5000)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P15_TFIDF_ComplementNB": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", ComplementNB(alpha=0.5)),
    ]),
})

In [5]:
resultats = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    metrics = fit_evaluate_and_log_sklearn_pipeline(
        name=nom_pipeline,
        estimator=pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        notebook_name="NB3",
        family_name="reduction_selection_nb",
        output_dir=RESULTS_DIR,
        log_model=MLFLOW_LOG_MODEL,
    )
    resultats.append(metrics)

results_df = round_results(pd.DataFrame(resultats))
display(results_df)


Entraînement -> P11_TFIDF_SVD_LogReg


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('svd', TruncatedSVD(n_components=300, random_state=42)),
                ('norm', Normalizer(copy=False)),
                ('clf', LogisticRegression(max_iter=2500))])

--------------------------------------------------------------------------------
Entraînement -> P12_TFIDF_SVD_LinearSVC


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('svd', TruncatedSVD(n_components=300, random_state=42)),
                ('norm', Normalizer(copy=False)), ('clf', LinearSVC())])

--------------------------------------------------------------------------------
Entraînement -> P13_TFIDF_SelectKBest_LogReg


...

Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('select',
                 SelectKBest(k=5000,
                             score_func=<function chi2 at 0x00000232257C59E0>)),
                ('clf', LinearSVC())])

--------------------------------------------------------------------------------
Entraînement -> P15_TFIDF_ComplementNB


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('clf', ComplementNB(alpha=0.5))])

--------------------------------------------------------------------------------


pipeline,P11_TFIDF_SVD_LogReg,P12_TFIDF_SVD_LinearSVC,P13_TFIDF_SelectKBest_LogReg,P14_TFIDF_SelectKBest_LinearSVC,P15_TFIDF_ComplementNB
train_accuracy,0.8830,0.8884,0.8883,0.9594,0.9454
train_precision_macro,0.8595,0.8570,0.9278,0.9666,0.9101
train_recall_macro,0.7224,0.7448,0.7037,0.8982,0.9092
train_f1_macro,0.7644,0.7833,0.7563,0.9278,0.9096
train_precision_weighted,0.8782,0.8828,0.8984,0.9601,0.9453
train_recall_weighted,0.8830,0.8884,0.8883,0.9594,0.9454
train_f1_weighted,0.8694,0.8781,0.8690,0.9578,0.9453
train_precision_class_0,0.8893,0.8981,0.8810,0.9563,0.9661
train_recall_class_0,0.9781,0.9734,0.9976,0.9957,0.9668
train_f1_class_0,0.9316,0.9342,0.9357,0.9756,0.9665


In [6]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans {RESULTS_DIR}")

Fichiers CSV/XLSX enregistrés dans ../../outputs/NB3
